# Prepare time series

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

In [ ]:
from etdataset.api import (
    parse_date,
    prepare_daily_et_timeseries,
    prepare_daily_radiation_timeseries,
)
from etdataset.era5 import download_date_by_date as download_et
from etdataset.et import (
    create_daily_et_dataset,
    create_daily_explanatory_dataset,
)
from etdataset.interpolation import create_grid_dataset
from etdataset.msg import create_daily_radiation_dataset
from etdataset.msg import download_date_by_date as download_rad
from etdataset.utils import get_utm_bbox_from_roi

In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

# Daily radiation

### Input

In [ ]:
# Acquisition dates
min_date_str = "2023-03-01"  # Format YYYY-MM-DD
max_date_str = "2023-03-10"  # Format YYYY-MM-DD

In [ ]:
min_date = parse_date(min_date_str)
max_date = parse_date(max_date_str)

In [ ]:
roi_path = os.path.join("data", "Zone_Senegal_Centre.shp")

In [ ]:
roi_bbox, roi_crs = get_utm_bbox_from_roi(roi_path)
grid = create_grid_dataset(roi_bbox, roi_crs, 3000)

In [ ]:
resolution = 3000

In [ ]:
output_path = "out_ts"
os.makedirs(output_path, exist_ok=True)
radiation_path = os.path.join(output_path, "timeseries/daily_radiation")
os.makedirs(radiation_path, exist_ok=True)
et_path = os.path.join(output_path, "timeseries/et")
os.makedirs(et_path, exist_ok=True)
extra_path = os.path.join(output_path, "timeseries/extra")
os.makedirs(extra_path, exist_ok=True)

### Downloading MSG data for the date range

In [ ]:
date_list_rad = download_rad(min_date, max_date, roi_bbox, roi_crs, output_path)

### Selecting one date to play with

In [ ]:
date_rad = date_list_rad[0]

### Reading the corresponding .nc file as dataset

In [ ]:
dst_rad = create_daily_radiation_dataset(date_rad, grid, output_path)

In [ ]:
dst_rad

### Plotting the spatial distribution of daily radiation over the ROI

In [ ]:
dst_rad.daily_radiation.plot()

### Preparing daily radidation files for the entire date range

In [ ]:
prepare_daily_radiation_timeseries(
    start_date=min_date_str,
    end_date=max_date_str,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    resolution=resolution,
    output=output_path,
)

# ET Single Date

### Downloading ERA5-Land "Total Evaporation" product for the date range

In [ ]:
date_list_et = download_et(
    min_date,
    max_date,
    [
        "total_evaporation",
        "total_precipitation",
        "surface_runoff",
        "skin_reservoir_content",
        "volumetric_soil_water_layer_1",
    ],
    output_path,
)

In [ ]:
date_et = date_list_et[1]

### Reading the content of the .zip directory as dataset

In [ ]:
dst_et = create_daily_et_dataset(date_et, grid, output_path)

In [ ]:
dst_et

### Plotting the spatial distribution of evaporation over the ROI

In [ ]:
dst_et.et.plot(cmap="magma")

### Reading the content of the .zip directory as dataset

In [ ]:
dst_extra = create_daily_explanatory_dataset(date_et, grid, output_path)

In [ ]:
dst_extra

### Plotting the spatial distribution of the variables over the ROI

In [ ]:
dst_extra.tp.plot()

In [ ]:
dst_extra.sro.plot()

In [ ]:
dst_extra.src.plot()

In [ ]:
dst_extra.sw.plot()

### Preparing et single date files for the entire date range

In [ ]:
prepare_daily_et_timeseries(
    start_date=min_date_str,
    end_date=max_date_str,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    resolution=3000,
    output=output_path,
    extra_variables=True,
)